# 🗄️ SQL for Data Analysis

**Goal:** learn SQL by querying a real database — the same SQL used in data analyst jobs.
We will **build** a small shop database with Python, then answer business questions with SQL.

In [1]:
import os
import sqlite3
import pandas as pd
import random
from datetime import date, timedelta

random.seed(42)  # same data every run

# Start fresh each run so the notebook is re-runnable
if os.path.exists("data/shop.db"):
    os.remove("data/shop.db")

conn = sqlite3.connect("data/shop.db")
cur = conn.cursor()
print("Connected to data/shop.db")

Connected to data/shop.db


## Step 1: Create the tables

Three tables, like a real shop system:
- **customers** — who buys
- **products** — what is sold
- **orders** — who bought what, and when

In [2]:
cur.execute('''CREATE TABLE customers (
    CustomerID INTEGER PRIMARY KEY,
    Name TEXT,
    City TEXT,
    JoinDate TEXT)''')

cur.execute('''CREATE TABLE products (
    ProductID INTEGER PRIMARY KEY,
    ProductName TEXT,
    Category TEXT,
    Price REAL)''')

cur.execute('''CREATE TABLE orders (
    OrderID INTEGER PRIMARY KEY,
    CustomerID INTEGER,
    ProductID INTEGER,
    Quantity INTEGER,
    OrderDate TEXT)''')

conn.commit()
print("Tables created: customers, products, orders")

Tables created: customers, products, orders


## Step 2: Generate data and insert it

200 customers, 10 products, 1000 orders — inserted with `executemany` (fast bulk insert).
Note: orders only use customers 1–800, so 200 customers never ordered (we will find them with SQL later!).

In [3]:
first = ["Ahmed","Sara","Bilal","Fatima","Usman","Ayesha","Ali","Hassan",
         "Mahnoor","Danish","Sana","Imran","Zainab","Fahad","Hira",
         "Kamran","Rabia","Tariq","Nida","Omar"]
last = ["Khan","Ahmed","Ali","Hussain","Malik","Shah","Raza","Butt","Chaudhry","Farooq"]
cities = ["Lahore","Karachi","Islamabad","Peshawar","Quetta","Multan","Faisalabad","Rawalpindi"]

def rand_date(start, end):
    # random date between two dates
    delta = (end - start).days
    return (start + timedelta(days=random.randint(0, delta))).isoformat()

customers = []
for i in range(1, 1001):
    name = random.choice(first) + " " + random.choice(last)
    city = random.choice(cities)
    join = rand_date(date(2024, 1, 1), date(2025, 12, 31))
    customers.append((i, name, city, join))

products = [
    (1, "Laptop", "Electronics", 150000),
    (2, "Monitor", "Electronics", 35000),
    (3, "Keyboard", "Electronics", 4500),
    (4, "Mouse", "Electronics", 2500),
    (5, "Headphones", "Electronics", 8000),
    (6, "Desk Chair", "Furniture", 22000),
    (7, "Office Desk", "Furniture", 45000),
    (8, "Notebook Pack", "Stationery", 800),
    (9, "Pen Set", "Stationery", 1200),
    (10, "Printer", "Electronics", 28000),
    (11, "Tablet", "Electronics", 55000),
    (12, "Smartphone", "Electronics", 85000),
    (13, "Smartwatch", "Electronics", 18000),
    (14, "Bookshelf", "Furniture", 15000),
    (15, "Filing Cabinet", "Furniture", 12000),
    (16, "Desk Lamp", "Furniture", 3500),
    (17, "Sticky Notes", "Stationery", 400),
    (18, "Highlighter Set", "Stationery", 900),
    (19, "Calculator", "Stationery", 2500),
    (20, "Webcam", "Electronics", 6500),
]

orders = []
for i in range(1, 10001):
    cust = random.randint(1, 800)   # only first 800 customers order
    prod = random.randint(1, 20)
    qty = random.randint(1, 5)
    odate = rand_date(date(2026, 1, 1), date(2026, 6, 30))
    orders.append((i, cust, prod, qty, odate))

cur.executemany("INSERT INTO customers VALUES (?,?,?,?)", customers)
cur.executemany("INSERT INTO products VALUES (?,?,?,?)", products)
cur.executemany("INSERT INTO orders VALUES (?,?,?,?,?)", orders)
conn.commit()

print("customers:", cur.execute("SELECT COUNT(*) FROM customers").fetchone()[0])
print("products:", cur.execute("SELECT COUNT(*) FROM products").fetchone()[0])
print("orders:", cur.execute("SELECT COUNT(*) FROM orders").fetchone()[0])

customers: 1000
products: 20
orders: 10000


## Concept 1 — SELECT, WHERE, ORDER BY, LIMIT

- `SELECT` picks columns, `WHERE` filters rows,
  `ORDER BY` sorts, `LIMIT` keeps only the first N rows.

In [4]:
q = '''SELECT ProductName, Category, Price
       FROM products
       WHERE Price > 30000
       ORDER BY Price DESC
       LIMIT 5'''
pd.read_sql_query(q, conn)

,ProductName,Category,Price
0,Laptop,Electronics,150000.0
1,Smartphone,Electronics,85000.0
2,Tablet,Electronics,55000.0
3,Office Desk,Furniture,45000.0
4,Monitor,Electronics,35000.0


## Concept 2 — Aggregations: COUNT, SUM, AVG + GROUP BY + HAVING

`GROUP BY` splits rows into groups, then COUNT/SUM/AVG summarize each group.
`HAVING` filters **groups** (WHERE cannot do that).

In [5]:
q = '''SELECT City,
              COUNT(*) AS num_customers
       FROM customers
       GROUP BY City
       HAVING COUNT(*) >= 20
       ORDER BY num_customers DESC'''
pd.read_sql_query(q, conn)

,City,num_customers
0,Karachi,140
1,Multan,137
2,Quetta,125
3,Faisalabad,123
4,Islamabad,121
5,Lahore,120
6,Peshawar,119
7,Rawalpindi,115


## Concept 3 — INNER JOIN

`JOIN` combines tables using a shared key.
Here we link orders → customers → products to see full order details.

In [6]:
q = '''SELECT c.Name AS customer, p.ProductName AS product,
              o.Quantity, o.OrderDate
       FROM orders o
       INNER JOIN customers c ON o.CustomerID = c.CustomerID
       INNER JOIN products p ON o.ProductID = p.ProductID
       LIMIT 5'''
pd.read_sql_query(q, conn)

,customer,product,Quantity,OrderDate
0,Mahnoor Khan,Desk Chair,3,2026-06-12
1,Fahad Ali,Laptop,4,2026-03-12
2,Danish Farooq,Desk Chair,5,2026-02-10
3,Nida Shah,Keyboard,2,2026-06-07
4,Sana Ali,Bookshelf,4,2026-04-20


## Concept 4 — LEFT JOIN

`LEFT JOIN` keeps **all** left-table rows, even with no match.
`WHERE ... IS NULL` then finds customers who **never** ordered.

In [7]:
q = '''SELECT c.CustomerID, c.Name, c.City
       FROM customers c
       LEFT JOIN orders o ON c.CustomerID = o.CustomerID
       WHERE o.OrderID IS NULL
       LIMIT 10'''
pd.read_sql_query(q, conn)

,CustomerID,Name,City
0,801,Ali Ali,Multan
1,802,Nida Ahmed,Quetta
2,803,Fahad Hussain,Lahore
3,804,Bilal Raza,Karachi
4,805,Omar Farooq,Lahore
5,806,Ayesha Ahmed,Lahore
6,807,Usman Khan,Islamabad
7,808,Imran Chaudhry,Quetta
8,809,Imran Ali,Quetta
9,810,Ahmed Shah,Faisalabad


## Concept 5 — Subquery

A query **inside** another query. The inner query runs first.
Here: products priced above the average product price.

In [8]:
q = '''SELECT ProductName, Price
       FROM products
       WHERE Price > (SELECT AVG(Price) FROM products)
       ORDER BY Price DESC'''
pd.read_sql_query(q, conn)

,ProductName,Price
0,Laptop,150000.0
1,Smartphone,85000.0
2,Tablet,55000.0
3,Office Desk,45000.0
4,Monitor,35000.0
5,Printer,28000.0


## Concept 6 — Window function: RANK()

`RANK() OVER (ORDER BY ...)` ranks rows **without** collapsing them into groups.
Here: rank customers by total spending.

In [9]:
q = '''SELECT Name, total_spent,
              RANK() OVER (ORDER BY total_spent DESC) AS spending_rank
       FROM (SELECT c.Name AS Name,
                    SUM(o.Quantity * p.Price) AS total_spent
             FROM customers c
             JOIN orders o ON c.CustomerID = o.CustomerID
             JOIN products p ON o.ProductID = p.ProductID
             GROUP BY c.CustomerID)
       LIMIT 10'''
pd.read_sql_query(q, conn)

,Name,total_spent,spending_rank
0,Rabia Ali,3026400.0,1
1,Usman Chaudhry,3001500.0,2
2,Bilal Shah,2768700.0,3
3,Sana Ahmed,2704000.0,4
4,Tariq Chaudhry,2660900.0,5
5,Bilal Raza,2594700.0,6
6,Bilal Khan,2510200.0,7
7,Rabia Farooq,2503600.0,8
8,Ali Farooq,2439800.0,9
9,Fahad Ahmed,2411600.0,10


## 💼 Business questions

Now let's answer 5 real business questions with SQL.

**Q1: What is the total revenue?**

In [10]:
q = '''SELECT ROUND(SUM(o.Quantity * p.Price), 2) AS total_revenue
       FROM orders o
       JOIN products p ON o.ProductID = p.ProductID'''
pd.read_sql_query(q, conn)

,total_revenue
0,735993400.0


**Q2: Who are the top 5 customers by spending?**

In [11]:
q = '''SELECT c.Name, c.City,
              ROUND(SUM(o.Quantity * p.Price), 2) AS total_spent
       FROM customers c
       JOIN orders o ON c.CustomerID = o.CustomerID
       JOIN products p ON o.ProductID = p.ProductID
       GROUP BY c.CustomerID
       ORDER BY total_spent DESC
       LIMIT 5'''
pd.read_sql_query(q, conn)

,Name,City,total_spent
0,Rabia Ali,Multan,3026400.0
1,Usman Chaudhry,Lahore,3001500.0
2,Bilal Shah,Faisalabad,2768700.0
3,Sana Ahmed,Multan,2704000.0
4,Tariq Chaudhry,Islamabad,2660900.0


**Q3: What is the monthly revenue trend?**
(`substr(OrderDate, 1, 7)` extracts the "YYYY-MM" part of the date.)

In [12]:
q = '''SELECT substr(o.OrderDate, 1, 7) AS month,
              ROUND(SUM(o.Quantity * p.Price), 2) AS revenue
       FROM orders o
       JOIN products p ON o.ProductID = p.ProductID
       GROUP BY month
       ORDER BY month'''
pd.read_sql_query(q, conn)

,month,revenue
0,2026-01,134117100.0
1,2026-02,109713400.0
2,2026-03,123016200.0
3,2026-04,126345400.0
4,2026-05,127943000.0
5,2026-06,114858300.0


**Q4: Which product category sells best?**

In [13]:
q = '''SELECT p.Category,
              SUM(o.Quantity) AS units_sold,
              ROUND(SUM(o.Quantity * p.Price), 2) AS revenue
       FROM orders o
       JOIN products p ON o.ProductID = p.ProductID
       GROUP BY p.Category
       ORDER BY revenue DESC'''
pd.read_sql_query(q, conn)

,Category,units_sold,revenue
0,Electronics,15155,584318000.0
1,Furniture,7402,142880000.0
2,Stationery,7379,8795400.0


**Q5: Which customers never ordered?**

In [14]:
q = '''SELECT COUNT(*) AS customers_never_ordered
       FROM customers c
       LEFT JOIN orders o ON c.CustomerID = o.CustomerID
       WHERE o.OrderID IS NULL'''
pd.read_sql_query(q, conn)

,customers_never_ordered
0,200


## ✅ Done!

You learned: SELECT/WHERE/ORDER BY/LIMIT → aggregations + GROUP BY/HAVING →
INNER JOIN → LEFT JOIN → subqueries → window functions —
and answered 5 real business questions. This is exactly what data analysts do daily.